# 03 — Error analysis and label-fix audit

Reads the decision tables produced by notebook 02 (`evaluate.py`). It lists every error for each method and compares the audit output before and after the label fix recorded in `FAILURE_LOG.md`.

Run notebook 02 first, so the CSVs exist.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import platform, torch, networkx, sklearn, matplotlib
print("python", platform.python_version(), "| torch", torch.__version__,
      "| networkx", networkx.__version__, "| scikit-learn", sklearn.__version__,
      "| matplotlib", matplotlib.__version__)

python 3.13.3 | torch 2.13.0+cpu | networkx 3.6.1 | scikit-learn 1.9.0 | matplotlib 3.11.1


## 1. Errors per method (leave-one-out, 18 decisions each)

In [2]:
import pandas as pd
gat = pd.read_csv("results/loco_gat_decisions.csv")
gat_seed0 = gat[gat["seed"] == 0]
sink = pd.read_csv("results/loco_sink_baseline_decisions.csv")
logi = pd.read_csv("results/loco_logistic_baseline_decisions.csv")

def errors(df, pred_col="pred"):
    return df[df["truth"] != df[pred_col]]

print("GAT (seed 0) errors:")
display(errors(gat_seed0)[["contract","class","truth","prob","pred"]])
print("Sink-presence rule errors:")
display(errors(sink)[["contract","class","truth","pred"]])
print("Logistic regression errors:")
display(errors(logi)[["contract","class","truth","prob","pred"]])

GAT (seed 0) errors:


,contract,class,truth,prob,pred
0,ReentrancyVulnerable.sol,Reentrancy,1,0.000004,0
3,ReentrancySafe.sol,Reentrancy,0,0.959808,1
7,TimestampVulnerable.sol,Timestamp Dependency,1,0.000025,0
14,ArithmeticVulnerable.sol,Arithmetic Error,1,0.000045,0


Sink-presence rule errors:


,contract,class,truth,pred
2,ReentrancyVulnerable.sol,Arithmetic Error,0,1
3,ReentrancySafe.sol,Reentrancy,0,1
5,ReentrancySafe.sol,Arithmetic Error,0,1
12,ArithmeticVulnerable.sol,Reentrancy,0,1
15,ArithmeticSafe.sol,Reentrancy,0,1
17,ArithmeticSafe.sol,Arithmetic Error,0,1


Logistic regression errors:


,contract,class,truth,prob,pred
0,ReentrancyVulnerable.sol,Reentrancy,1,0.0,0
7,TimestampVulnerable.sol,Timestamp Dependency,1,0.0,0
14,ArithmeticVulnerable.sol,Arithmetic Error,1,0.0,0


## 2. GAT errors are stable across seeds

Each cell is the GAT's error count over the five seeds for that contract and class.

In [3]:
err = gat[gat["truth"] != gat["pred"]]
print(err.groupby(["contract","class"]).size().rename("errors out of 5 seeds").to_string())

contract                  class               
ArithmeticVulnerable.sol  Arithmetic Error        5
ReentrancySafe.sol        Reentrancy              5
ReentrancyVulnerable.sol  Reentrancy              5
TimestampVulnerable.sol   Timestamp Dependency    5


## 3. Why the held-out Reentrancy decision fails

When `ReentrancyVulnerable.sol` is held out, the only Reentrancy training example is `ReentrancySafe.sol`, which is labelled safe. When `ReentrancySafe.sol` is held out, the only positive example is `ReentrancyVulnerable.sol`. In both cases the classifier is asked to generalize from one example per label. The table shows that the GAT's probability for the held-out safe contract is high, which is the opposite of what the label says.

In [4]:
print(gat_seed0[(gat_seed0["class"] == "Reentrancy")][["contract","truth","prob","pred"]].to_string(index=False))

                contract  truth     prob  pred
ReentrancyVulnerable.sol      1 0.000004     0
      ReentrancySafe.sol      0 0.959808     1
 TimestampVulnerable.sol      0 0.000197     0
       TimestampSafe.sol      0 0.060155     0
ArithmeticVulnerable.sol      0 0.000359     0
      ArithmeticSafe.sol      0 0.003530     0


## 4. Label-fix audit (before and after)

The training labels for the Timestamp and Arithmetic contracts pointed at the wrong lines (10 and 14 instead of 9 and 12). The table compares the localized lines and per-class confidence from the audit runs before the fix (`results/logs/before_fixes/`) and after it (`results/logs/`).

In [5]:
import re, glob, os
def summarize(path):
    text = open(path, encoding="utf-8").read()
    conf = dict(re.findall(r"\[([A-Za-z ]+)\] Confidence: ([0-9.]+)%", text))
    lines = re.findall(r"\[Line (\d+)\]", text)
    return conf, lines

rows = []
for name in ["ReentrancyVulnerable", "TimestampVulnerable", "ArithmeticVulnerable"]:
    before_c, before_l = summarize(f"results/logs/before_fixes/auditor_{name}.txt")
    after_c, after_l = summarize(f"results/logs/audit_{name}.txt")
    rows.append({"contract": name,
                 "before_lines": " ".join(before_l) or "none",
                 "after_lines": " ".join(after_l) or "none",
                 "before_top_conf": max(before_c.values(), key=float) if before_c else "",
                 "after_top_conf": max(after_c.values(), key=float) if after_c else ""})
display(pd.DataFrame(rows))

,contract,before_lines,after_lines,before_top_conf,after_top_conf
0,ReentrancyVulnerable,015 016,015 016,99.10,58.29
1,TimestampVulnerable,none,009,99.83,99.92
2,ArithmeticVulnerable,none,none,99.73,99.88


## 5. Other observations

* The Arithmetic vulnerable contract is sliced down to 6 nodes (see `results/slicing_sizes.csv`). The remaining graph has little context, and the model localizes nothing for it.
* Keyword features match substrings such as `"now"` and `"send"` anywhere in a node's text. That is one plausible source of the false signals above, but it has not been isolated by an ablation.